# 03. Walk-Forward Strategy (WF1, WF2)

**WF1** re-selects pairs every 6 months using the corrected point-in-time screen (notebook 02),
trading each selection with S0's exact execution engine (frozen formation alpha/beta/time
stop). **WF2** is WF1 plus rolling entry-time hedge-ratio sizing -- the Z-score signal itself
still uses the frozen formation alpha/beta throughout; only entry-time position sizing is
rolling. Do not describe WF2 as a rolling-spread model.

**Caveat stated up front:** WF1/WF2 trade very few times in the full OOS window (see results
below) -- any comparison between them, or against S0/S1, must state this sample-size limitation
rather than imply a robust conclusion.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
for _p in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.research_config import (CONFIG, WF_CONFIG, MAX_HALF_LIFE_BARS, MIN_CROSSINGS, MODEL_IDS,
                                     S1_TIME_STOP_MULTIPLIER, S1_TIME_STOP_MULTIPLIER_ROBUSTNESS,
                                     FORWARD_EVAL_START)

In [2]:
from stat_arb.data import build_walkforward_universe
from stat_arb.walkforward import generate_rebalance_dates

WF_PX, WF_RET = build_walkforward_universe(WF_CONFIG)
REBALANCE_DATES = generate_rebalance_dates(CONFIG['FORMATION_START'], CONFIG['OOS_END'],
                                            WF_CONFIG['REBALANCE_MONTHS_PRIMARY'], WF_CONFIG['FORMATION_MONTHS'])
WF1_REBALANCE_DATES = [d for d in REBALANCE_DATES if d >= pd.Timestamp(CONFIG['OOS_START'])]
print(f'WF1 rebalance dates within the OOS trading window: {[d.date() for d in WF1_REBALANCE_DATES]}')

WF1 rebalance dates within the OOS trading window: [datetime.date(2024, 1, 1), datetime.date(2024, 7, 1), datetime.date(2025, 1, 1), datetime.date(2025, 7, 1), datetime.date(2026, 1, 1)]


## WF1 Pair Selection

Uses the exact same `screen_pairs_at_date`/`select_pairs_at_rebalance` as notebook 02's
diagnostic screen -- this section's selected-pair output is not a new methodology, it is the
trading decision built on that exact diagnostic. Cached to disk (methodology-aware
fingerprint) since re-screening 5 rebalance dates is itself nontrivially expensive.

In [3]:
from stat_arb.cache import wf_cache_fingerprint, cache_path, load_or_assert
from stat_arb.walkforward import run_wf1_selection_pipeline

wf1_sel_fingerprint = wf_cache_fingerprint(
    universe_cols=WF_PX.columns, rebalance_dates=WF1_REBALANCE_DATES, wf_config=WF_CONFIG,
    extra=f"max_pairs={WF_CONFIG['MAX_PAIRS_PER_REBALANCE']},ts_mult={WF_CONFIG['TIME_STOP_MULTIPLIER_PRIMARY']}")
wf1_sel_cache_path = cache_path('wf1_selections', wf1_sel_fingerprint)

WF1_SELECTIONS = run_wf1_selection_pipeline(WF1_REBALANCE_DATES, WF_PX, WF_PX.columns.tolist(),
                                             WF_CONFIG, wf1_sel_cache_path,
                                             max_pairs=WF_CONFIG['MAX_PAIRS_PER_REBALANCE'])

for rebal_date in WF1_REBALANCE_DATES:
    selected = WF1_SELECTIONS[rebal_date]
    if selected:
        pairs_str = ', '.join(f"{s['x']}/{s['y']} (TS={s['time_stop_bars']})" for s in selected)
        print(f'{rebal_date.date()}: {len(selected)} pairs selected -> {pairs_str}')
    else:
        print(f'{rebal_date.date()}: 0 pairs selected -> block trades nothing (cash)')

2024-01-01: 1 pairs selected -> OMUSDT/LDOUSDT (TS=76)
2024-07-01: 0 pairs selected -> block trades nothing (cash)
2025-01-01: 0 pairs selected -> block trades nothing (cash)
2025-07-01: 0 pairs selected -> block trades nothing (cash)
2026-01-01: 0 pairs selected -> block trades nothing (cash)


## WF1 Portfolio Construction

Fixed 1/N sleeve per selected pair at each block; a rebalance block with zero selected pairs
trades nothing and contributes a flat (zero-return) period to the continuous equity curve --
correct, non-optimized behavior, not a data gap.

In [4]:
from stat_arb.walkforward import run_wf1_full

WF1_FULL_FRAME, WF1_ALL_RESULTS, WF1_ALL_TRADES = run_wf1_full(
    WF1_SELECTIONS, WF1_REBALANCE_DATES, CONFIG['OOS_END'], WF_PX, CONFIG)

assert WF1_FULL_FRAME.index.is_monotonic_increasing
assert not WF1_FULL_FRAME.index.duplicated().any()
assert WF1_FULL_FRAME.index[0] == pd.Timestamp(CONFIG['OOS_START'])
assert WF1_FULL_FRAME.index[-1] <= pd.Timestamp(CONFIG['OOS_END'])

print(f'WF1 continuous portfolio built: {len(WF1_FULL_FRAME)} bars, '
      f'{sum(len(t) for t in WF1_ALL_TRADES.values())} total trades across '
      f'{len({p for (_, p) in WF1_ALL_TRADES.keys()})} distinct pairs traded.')
print(f'Cumulative net return: {(1 + WF1_FULL_FRAME["net_ret"]).prod() - 1:.4%}')
print(f'Time in market (portfolio gross_exposure > 0): {WF1_FULL_FRAME["active_flag"].mean():.2%}')

[NOTE] Peak realized gross_exposure = 1.192 at 2024-02-03 00:00:00 (intra-trade mark-to-market drift on fixed quantities, not a sleeve-allocation violation -- sleeve weights themselves sum to <=1.0 by construction).
WF1 continuous portfolio built: 5467 bars, 2 total trades across 1 distinct pairs traded.
Cumulative net return: -3.7954%
Time in market (portfolio gross_exposure > 0): 1.50%


In [5]:
from stat_arb.reporting import standardized_wf1_summary

WF1_ROW = standardized_wf1_summary(
    'WF1_SEMIANNUAL_REESTIMATION', WF1_FULL_FRAME, WF1_ALL_RESULTS, WF1_ALL_TRADES, WF_PX,
    CONFIG['OOS_START'], CONFIG['OOS_END'])
pd.Series(WF1_ROW)

model                      WF1_SEMIANNUAL_REESTIMATION
daily_sharpe_252                             -0.067867
ann_volatility                                0.093053
cum_net_return                               -0.037954
cum_gross_return                             -0.029919
max_drawdown                                 -0.185844
n_trades                                             2
n_pairs_traded                                       1
win_rate                                           0.5
avg_trade_return                             -0.012893
median_trade_return                          -0.012893
avg_holding_bars                                  41.0
median_holding_bars                               41.0
time_in_market_pct                            0.014999
turnover                                      4.152873
pct_exit_mean_reversion                            0.5
pct_exit_time_stop                                 0.0
pct_exit_hard_stop                                 0.5
total_cost

## Selection Funnel Diagnostic

**Research question: where does the opportunity set disappear between the eligible universe
and executed trades?** WF1 selected only 1 pair (OMUSDT/LDOUSDT, 2 trades) across the entire
OOS window despite notebook 02 showing 15 qualifying pairs at 2024-01-01 alone. This section
answers *why*, stage by stage, using the exact real pipeline (`screen_pairs_at_date` /
`select_pairs_at_rebalance`, instrumented additively via `collect_diagnostics=True` -- the same
code path that produced `WF1_SELECTIONS` above, not a second implementation) plus the REAL
trades already computed in `WF1_ALL_TRADES` -- nothing here is re-simulated or re-thresholded.

**Units, kept distinct throughout** (do not compare across these without noting the change of
unit): eligible **assets** -> candidate **unordered pairs** -> **testable** pairs (enough
overlapping observations) -> **directed** EG tests (2x testable pairs, both regression
directions) -> FDR-significant pairs **after direction resolution** (deduplicated to the
lower-adjusted-p-value direction) -> pairs passing positive-Beta -> Half-Life -> crossings, in
that exact order (matching `screen_pairs_at_date`'s own filtering loop) -> fully-qualifying
pairs -> selected pairs (one-pair-per-coin, rank-ordered by `pvalue_adj`, capped at
`MAX_PAIRS_PER_REBALANCE`) -> selected pairs with >=1 real entry signal -> executed trades.

This is precomputed by `scripts/run_selection_funnel.py` (expensive -- re-runs the point-in-time
screen with diagnostics enabled for every WF1 rebalance date) and cached to
`data/cache/selection_funnel_<fingerprint>.pkl`, keyed by the same `wf_cache_fingerprint`
convention as every other walk-forward cache (this does not add a new fingerprinting
methodology, and does not touch the existing `wf1_selections`/`pair_stability` caches or their
fingerprints). This cell only loads it.

In [6]:
funnel_extra = f"max_pairs={WF_CONFIG['MAX_PAIRS_PER_REBALANCE']},ts_mult={WF_CONFIG['TIME_STOP_MULTIPLIER_PRIMARY']}"
funnel_fingerprint = wf_cache_fingerprint(universe_cols=WF_PX.columns, rebalance_dates=WF1_REBALANCE_DATES,
                                           wf_config=WF_CONFIG, extra=f'{funnel_extra},funnel_diagnostics')
funnel_cache_path = cache_path('selection_funnel', funnel_fingerprint)
funnel_result = load_or_assert(
    funnel_cache_path,
    f'Selection funnel cache not found at {funnel_cache_path}. Regenerate via scripts/run_selection_funnel.py.')
FUNNEL_DF, FUNNEL_PAIR_LEVEL_DF, FUNNEL_OVERLAP_CONFLICTS_DF = (
    funnel_result['funnel_df'], funnel_result['pair_level_df'], funnel_result['conflicts_df'])

print('Selection funnel by rebalance date (see notebook markdown above for unit definitions):')
FUNNEL_DF.set_index('rebalance_date')

Selection funnel by rebalance date (see notebook markdown above for unit definitions):


,n_eligible_assets,n_candidate_pairs,n_testable_pairs,n_directed_eg_tests,n_fdr_significant_pairs,n_pass_beta,n_pass_half_life,n_pass_crossings,n_fully_qualifying,n_selected,n_removed_by_overlap,n_removed_by_max_pairs_cap,n_selected_with_entry_signal,n_selected_without_entry_signal,n_entry_events,n_executed_trades
rebalance_date,,,,,,,,,,,,,,,,
2024-01-01,33,528,528,1056,15,15,15,15,15,1,14,0,1,0,2,2
2024-07-01,36,630,630,1260,0,0,0,0,0,0,0,0,0,0,0,0
2025-01-01,39,741,741,1482,0,0,0,0,0,0,0,0,0,0,0,0
2025-07-01,41,820,820,1640,0,0,0,0,0,0,0,0,0,0,0,0
2026-01-01,42,861,861,1722,0,0,0,0,0,0,0,0,0,0,0,0


### One-pair-per-coin diagnostic

At 2024-01-01, every one of the 14 non-selected fully-qualifying pairs is eliminated purely by
the one-pair-per-coin constraint (`n_removed_by_overlap` in the table above), not by any
statistical filter -- `n_pass_beta`/`n_pass_half_life`/`n_pass_crossings` all equal
`n_fdr_significant_pairs` (15) at that date: **zero pairs were eliminated by Beta, Half-Life,
or crossing requirements.** The table below shows exactly which coin caused each conflict, the
already-selected pair that won it, and the ranking metric (`pvalue_adj`, ascending -- the
existing rule, unchanged) that decided it.

In [7]:
conflicts_2024_01 = FUNNEL_OVERLAP_CONFLICTS_DF[FUNNEL_OVERLAP_CONFLICTS_DF['rebalance_date'] == WF1_REBALANCE_DATES[0]]
print(f"{len(conflicts_2024_01)} pairs rejected by coin overlap at {WF1_REBALANCE_DATES[0].date()}; "
      f"conflicting coin(s): {sorted(conflicts_2024_01['conflicting_coin'].unique())}")
conflicts_2024_01[['rejected_pair', 'rejected_pvalue_adj', 'conflicting_coin', 'winning_pair', 'winning_pvalue_adj']]

14 pairs rejected by coin overlap at 2024-01-01; conflicting coin(s): ['LDOUSDT']


,rejected_pair,rejected_pvalue_adj,conflicting_coin,winning_pair,winning_pvalue_adj
0,"(LDOUSDT, NEARUSDT)",0.000885,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
1,"(DOTUSDT, LDOUSDT)",0.000885,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
2,"(LDOUSDT, UNIUSDT)",0.000885,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
3,"(FTMUSDT, LDOUSDT)",0.000885,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
4,"(ATOMUSDT, LDOUSDT)",0.000885,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
5,"(LDOUSDT, VETUSDT)",0.002648,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
6,"(DOGEUSDT, LDOUSDT)",0.002648,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
7,"(FILUSDT, LDOUSDT)",0.004610,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
8,"(LDOUSDT, SHIBUSDT)",0.006083,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358
9,"(ICPUSDT, LDOUSDT)",0.006242,LDOUSDT,"(LDOUSDT, OMUSDT)",0.000358


### Pair-level bottleneck table (2024-01-01 -- the richest rebalance window)

One row per testable pair at this date: pass/fail through every stage, in the actual
implementation order, and the first stage at which each pair was eliminated (`None` only for
the one pair that was selected AND generated an executed trade).

In [8]:
pair_level_2024_01 = FUNNEL_PAIR_LEVEL_DF[FUNNEL_PAIR_LEVEL_DF['rebalance_date'] == WF1_REBALANCE_DATES[0]].copy()
pair_level_2024_01['pair'] = pair_level_2024_01['pair'].apply(lambda p: f'{p[0]}/{p[1]}')
display_cols = ['pair', 'pvalue_adj', 'passed_fdr', 'passed_beta', 'passed_half_life', 'passed_crossings',
                'fully_qualifying', 'selected', 'generated_entry_signal', 'n_executed_trades', 'elimination_stage']
pair_level_2024_01[display_cols].sort_values('pvalue_adj').reset_index(drop=True)

,pair,pvalue_adj,passed_fdr,passed_beta,passed_half_life,passed_crossings,fully_qualifying,selected,generated_entry_signal,n_executed_trades,elimination_stage
0,LDOUSDT/OMUSDT,0.000358,True,True,True,True,True,True,True,2.0,None
1,DOTUSDT/LDOUSDT,0.000885,True,True,True,True,True,False,NaN,NaN,coin_overlap
2,FTMUSDT/LDOUSDT,0.000885,True,True,True,True,True,False,NaN,NaN,coin_overlap
3,LDOUSDT/UNIUSDT,0.000885,True,True,True,True,True,False,NaN,NaN,coin_overlap
4,LDOUSDT/NEARUSDT,0.000885,True,True,True,True,True,False,NaN,NaN,coin_overlap
...,...,...,...,...,...,...,...,...,...,...,...
523,RUNEUSDT/VETUSDT,0.995786,False,NaN,NaN,NaN,False,False,NaN,NaN,fdr
524,IMXUSDT/RUNEUSDT,0.995786,False,NaN,NaN,NaN,False,False,NaN,NaN,fdr
525,MKRUSDT/VETUSDT,0.995786,False,NaN,NaN,NaN,False,False,NaN,NaN,fdr
526,IMXUSDT/MKRUSDT,0.995786,False,NaN,NaN,NaN,False,False,NaN,NaN,fdr


### Interpretation

At 2024-01-01, the opportunity set does **not** shrink for lack of statistically valid
relationships (528 candidate pairs -> 15 fully-qualifying, all surviving FDR/Beta/Half-Life/
crossings intact) -- it shrinks almost entirely at the **one-pair-per-coin portfolio
constraint**, driven by **concentration around a single coin (LDOUSDT)**: 14 of the 15
qualifying pairs involve LDOUSDT paired with a different counterpart, so once the
lowest-adjusted-p-value pair (OMUSDT/LDOUSDT) claims that coin, every other LDOUSDT pair is
mechanically excluded regardless of its own statistical strength. The `MAX_PAIRS_PER_REBALANCE`
cap played no role at this date (`n_removed_by_max_pairs_cap = 0`) -- overlap alone did all the
work. At every other WF1 rebalance date, zero pairs are even FDR-significant, so there is
nothing to select from at all -- a completely different bottleneck (statistical, not portfolio).
Execution/state-machine constraints did not reduce the opportunity set at 2024-01-01 either:
the one selected pair (OMUSDT/LDOUSDT) generated a real entry signal and both of its signal-
level trades were executed, with no drop between "entry signal" and "executed trade" -- in this
fixed-quantity, next-bar-execution engine those two counts are structurally identical (see
`stat_arb.funnel`'s module docstring).

### Active-Day BTC Beta (feedback #11)

Unconditional metrics can be misleading for low-activity strategies: WF1's unconditional beta
looks market-neutral largely because it is flat most days. Active-day beta (restricted to bars
with `gross_exposure > 0`) tells the real story on the days it actually trades.

In [9]:
from stat_arb.reporting import active_day_benchmark_stats, active_mask_from_bar_frame
from stat_arb.metrics import aggregate_daily_returns, build_btc_daily_benchmark

wf1_daily, _ = aggregate_daily_returns(WF1_FULL_FRAME['net_ret'])
wf1_active_daily_mask = active_mask_from_bar_frame(WF1_FULL_FRAME['active_flag']).reindex(wf1_daily.index, fill_value=False)
wf1_btc_daily = build_btc_daily_benchmark(wf1_daily.index, WF_PX)
wf1_active_bench = active_day_benchmark_stats(wf1_daily, wf1_active_daily_mask, wf1_btc_daily)

print(f"WF1 unconditional beta: {WF1_ROW['btc_beta']:.4f}   |   WF1 active-day beta: {wf1_active_bench['beta']:.4f}")

WF1 unconditional beta: 0.0141   |   WF1 active-day beta: 0.7809


### WF1 Robustness: time-stop multiplier and rebalance frequency

Pre-specified sensitivity checks only, per `CLAUDE.md` -- no grid search, no selection on OOS
performance.

In [10]:
def reselect_with_multiplier(base_selections, multiplier):
    out = {}
    for rebal_date, selected in base_selections.items():
        resel = []
        for s in selected:
            new_s = dict(s)
            new_s['time_stop_bars'] = int(np.ceil(multiplier * s['half_life_bars']))
            resel.append(new_s)
        out[rebal_date] = resel
    return out

WF1_MULTIPLIER_ROWS = []
for mult in S1_TIME_STOP_MULTIPLIER_ROBUSTNESS:
    sels_mult = reselect_with_multiplier(WF1_SELECTIONS, mult)
    frame_mult, results_mult, trades_mult = run_wf1_full(sels_mult, WF1_REBALANCE_DATES, CONFIG['OOS_END'], WF_PX, CONFIG)
    row = standardized_wf1_summary(f'WF1_x{mult}', frame_mult, results_mult, trades_mult, WF_PX,
                                    CONFIG['OOS_START'], CONFIG['OOS_END'])
    WF1_MULTIPLIER_ROWS.append(row)

pd.DataFrame(WF1_MULTIPLIER_ROWS).set_index('model')[['daily_sharpe_252', 'cum_net_return', 'n_trades']].round(4)

[NOTE] Peak realized gross_exposure = 1.166 at 2024-01-22 04:00:00 (intra-trade mark-to-market drift on fixed quantities, not a sleeve-allocation violation -- sleeve weights themselves sum to <=1.0 by construction).


[NOTE] Peak realized gross_exposure = 1.192 at 2024-02-03 00:00:00 (intra-trade mark-to-market drift on fixed quantities, not a sleeve-allocation violation -- sleeve weights themselves sum to <=1.0 by construction).


[NOTE] Peak realized gross_exposure = 1.192 at 2024-02-03 00:00:00 (intra-trade mark-to-market drift on fixed quantities, not a sleeve-allocation violation -- sleeve weights themselves sum to <=1.0 by construction).


,daily_sharpe_252,cum_net_return,n_trades
model,,,
WF1_x2.0,0.2746,0.0676,2
WF1_x2.5,-0.0679,-0.0380,2
WF1_x3.0,-0.0679,-0.0380,2


### WF1 Rebalance-Frequency Robustness (6-month primary vs. 3-month)

All other parameters (formation window, entry/exit/hard-stop thresholds, time-stop multiplier
=2.5x primary, universe, cost assumptions) are held fixed -- only the rebalance spacing changes.
Reuses the exact same `select_pairs_at_rebalance`/`run_wf1_full` pipeline as the primary 6-month
spec, just re-screened at a 3-month cadence -- no separate implementation.

In [11]:
WF1_REBAL_3M_DATES = pd.date_range(CONFIG['OOS_START'], CONFIG['OOS_END'],
                                    freq=f"{WF_CONFIG['REBALANCE_MONTHS_ROBUSTNESS']}MS")
WF1_REBAL_3M_DATES = [d for d in WF1_REBAL_3M_DATES if d >= pd.Timestamp(CONFIG['OOS_START'])]
print(f'3-month rebalance dates: {[d.date() for d in WF1_REBAL_3M_DATES]}')

wf1_3m_fingerprint = wf_cache_fingerprint(
    universe_cols=WF_PX.columns, rebalance_dates=WF1_REBAL_3M_DATES, wf_config=WF_CONFIG,
    extra=f"max_pairs={WF_CONFIG['MAX_PAIRS_PER_REBALANCE']},ts_mult={WF_CONFIG['TIME_STOP_MULTIPLIER_PRIMARY']}")
WF1_3M_SELECTIONS = run_wf1_selection_pipeline(WF1_REBAL_3M_DATES, WF_PX, WF_PX.columns.tolist(), WF_CONFIG,
                                                cache_path('wf1_selections_3m', wf1_3m_fingerprint),
                                                max_pairs=WF_CONFIG['MAX_PAIRS_PER_REBALANCE'])

for rebal_date, sel in WF1_3M_SELECTIONS.items():
    pairs_str = ', '.join(f"{s['x']}/{s['y']}" for s in sel) if sel else '(cash)'
    print(f'{rebal_date.date()}: {len(sel)} pairs -> {pairs_str}')

3-month rebalance dates: [datetime.date(2024, 1, 1), datetime.date(2024, 4, 1), datetime.date(2024, 7, 1), datetime.date(2024, 10, 1), datetime.date(2025, 1, 1), datetime.date(2025, 4, 1), datetime.date(2025, 7, 1), datetime.date(2025, 10, 1), datetime.date(2026, 1, 1), datetime.date(2026, 4, 1)]
2024-01-01: 1 pairs -> OMUSDT/LDOUSDT
2024-04-01: 0 pairs -> (cash)
2024-07-01: 0 pairs -> (cash)
2024-10-01: 1 pairs -> ARBUSDT/TIAUSDT
2025-01-01: 0 pairs -> (cash)
2025-04-01: 0 pairs -> (cash)
2025-07-01: 0 pairs -> (cash)
2025-10-01: 0 pairs -> (cash)
2026-01-01: 0 pairs -> (cash)
2026-04-01: 0 pairs -> (cash)


In [12]:
WF1_3M_FRAME, WF1_3M_RESULTS, WF1_3M_TRADES = run_wf1_full(
    WF1_3M_SELECTIONS, WF1_REBAL_3M_DATES, CONFIG['OOS_END'], WF_PX, CONFIG)

wf1_3m_row = standardized_wf1_summary('WF1_3MONTH', WF1_3M_FRAME, WF1_3M_RESULTS, WF1_3M_TRADES, WF_PX,
                                       CONFIG['OOS_START'], CONFIG['OOS_END'])
wf1_6m_row = standardized_wf1_summary('WF1_6MONTH_PRIMARY', WF1_FULL_FRAME, WF1_ALL_RESULTS, WF1_ALL_TRADES, WF_PX,
                                       CONFIG['OOS_START'], CONFIG['OOS_END'])

wf1_rebal_freq_df = pd.DataFrame([wf1_6m_row, wf1_3m_row]).set_index('model')
print('WF1 rebalance-frequency robustness -- OOS (all other parameters held fixed at primary spec):')
wf1_rebal_freq_df[['daily_sharpe_252', 'cum_net_return', 'max_drawdown', 'n_trades',
                    'n_pairs_traded', 'turnover']].round(4)

[NOTE] Peak realized gross_exposure = 1.192 at 2024-02-03 00:00:00 (intra-trade mark-to-market drift on fixed quantities, not a sleeve-allocation violation -- sleeve weights themselves sum to <=1.0 by construction).
WF1 rebalance-frequency robustness -- OOS (all other parameters held fixed at primary spec):


,daily_sharpe_252,cum_net_return,max_drawdown,n_trades,n_pairs_traded,turnover
model,,,,,,
WF1_6MONTH_PRIMARY,-0.0679,-0.038,-0.1858,2,1,4.1529
WF1_3MONTH,-0.0679,-0.038,-0.1858,2,1,4.1529


## B2.5 Ablation: Removing the One-Pair-Per-Coin Constraint (Exploratory)

**This is an exploratory Post-Sample Historical ablation, not a new primary model.** The
Selection Funnel Diagnostic above found that at 2024-01-01, all 15 fully-qualifying pairs
happened to involve LDOUSDT, and the one-pair-per-coin constraint let only 1 of them
(OMUSDT/LDOUSDT) trade. This section asks: what actually happens if that constraint is
relaxed? Exactly one assumption changes -- `select_pairs_at_rebalance(...,
enforce_one_pair_per_coin=False)` -- while every other function (ranking rule, `MAX_PAIRS_PER_REBALANCE`
cap, formation window, EG/FDR/Beta/Half-Life/crossing filters, execution engine, equal-sleeve
portfolio construction, cost model, benchmark regression) is exactly the same shared code used
by `WF1_SEMIANNUAL_REESTIMATION` (`stat_arb.walkforward.run_wf1_full`,
`stat_arb.reporting.standardized_wf1_summary`) -- nothing here is a second implementation.
This new variant, `WF1_NO_OVERLAP_CONSTRAINT`, is never substituted for WF1 in the master
model-comparison table and is not promoted to Notebook 04 as a primary result.

In [13]:
from stat_arb.ablation import run_no_overlap_selection_pipeline

no_overlap_extra = (f"max_pairs={WF_CONFIG['MAX_PAIRS_PER_REBALANCE']},"
                     f"ts_mult={WF_CONFIG['TIME_STOP_MULTIPLIER_PRIMARY']},no_overlap_constraint")
no_overlap_sel_fingerprint = wf_cache_fingerprint(universe_cols=WF_PX.columns, rebalance_dates=WF1_REBALANCE_DATES,
                                                    wf_config=WF_CONFIG, extra=no_overlap_extra)
NO_OVERLAP_SELECTIONS = run_no_overlap_selection_pipeline(
    WF1_REBALANCE_DATES, WF_PX, WF_PX.columns.tolist(), WF_CONFIG,
    cache_path('wf1_no_overlap_selections', no_overlap_sel_fingerprint),
    max_pairs=WF_CONFIG['MAX_PAIRS_PER_REBALANCE'])

for rebal_date in WF1_REBALANCE_DATES:
    print(f"{rebal_date.date()}: baseline {len(WF1_SELECTIONS[rebal_date])} pair(s) -> "
          f"no-overlap {len(NO_OVERLAP_SELECTIONS[rebal_date])} pair(s)")

2024-01-01: baseline 1 pair(s) -> no-overlap 10 pair(s)
2024-07-01: baseline 0 pair(s) -> no-overlap 0 pair(s)
2025-01-01: baseline 0 pair(s) -> no-overlap 0 pair(s)
2025-07-01: baseline 0 pair(s) -> no-overlap 0 pair(s)
2026-01-01: baseline 0 pair(s) -> no-overlap 0 pair(s)


At every rebalance date OTHER than 2024-01-01, zero pairs are even FDR-significant
(the same purely-statistical bottleneck the funnel diagnostic already found for WF1) -- removing
the overlap constraint cannot create trading opportunity where none survives the statistical
filters in the first place. Backtested with the exact same `run_wf1_full` used for WF1/WF2 (the
equal-sleeve portfolio construction is already generic over however many pairs a block selects,
so 10 simultaneously-selected pairs each automatically get a 1/10 sleeve, never 1/1 -- no new
capital-normalization logic was written for this ablation).

In [14]:
NO_OVERLAP_FULL_FRAME, NO_OVERLAP_ALL_RESULTS, NO_OVERLAP_ALL_TRADES = run_wf1_full(
    NO_OVERLAP_SELECTIONS, WF1_REBALANCE_DATES, CONFIG['OOS_END'], WF_PX, CONFIG)

NO_OVERLAP_ROW = standardized_wf1_summary(
    MODEL_IDS['WF1_NO_OVERLAP'], NO_OVERLAP_FULL_FRAME, NO_OVERLAP_ALL_RESULTS, NO_OVERLAP_ALL_TRADES,
    WF_PX, CONFIG['OOS_START'], CONFIG['OOS_END'])

ablation_comparison_df = pd.DataFrame([WF1_ROW, NO_OVERLAP_ROW]).set_index('model')
print('WF1 baseline vs. WF1_NO_OVERLAP_CONSTRAINT (exploratory ablation) -- OOS standardized comparison:')
ablation_comparison_df.round(4).T

WF1 baseline vs. WF1_NO_OVERLAP_CONSTRAINT (exploratory ablation) -- OOS standardized comparison:


model,WF1_SEMIANNUAL_REESTIMATION,WF1_NO_OVERLAP_CONSTRAINT
daily_sharpe_252,-0.0679,-0.4749
ann_volatility,0.0931,0.0583
cum_net_return,-0.0380,-0.1010
cum_gross_return,-0.0299,-0.0896
max_drawdown,-0.1858,-0.1291
n_trades,2.0000,31.0000
n_pairs_traded,1.0000,10.0000
win_rate,0.5000,0.4194
avg_trade_return,-0.0129,-0.0353
median_trade_return,-0.0129,-0.0156


### Active-Day BTC Beta, Both Models

Reuses `active_day_benchmark_stats` exactly as for WF1 above (`wf1_active_bench` was already
computed in that earlier cell).

In [15]:
no_overlap_daily, _ = aggregate_daily_returns(NO_OVERLAP_FULL_FRAME['net_ret'])
no_overlap_active_mask = active_mask_from_bar_frame(NO_OVERLAP_FULL_FRAME['active_flag']).reindex(
    no_overlap_daily.index, fill_value=False)
no_overlap_btc_daily = build_btc_daily_benchmark(no_overlap_daily.index, WF_PX)
no_overlap_active_bench = active_day_benchmark_stats(no_overlap_daily, no_overlap_active_mask, no_overlap_btc_daily)

print(f"WF1 baseline:          unconditional beta {WF1_ROW['btc_beta']:.4f} | "
      f"active-day beta {wf1_active_bench['beta']:.4f} (n={wf1_active_bench['n_obs']} active days)")
print(f"No-overlap ablation:   unconditional beta {NO_OVERLAP_ROW['btc_beta']:.4f} | "
      f"active-day beta {no_overlap_active_bench['beta']:.4f} (n={no_overlap_active_bench['n_obs']} active days)")

WF1 baseline:          unconditional beta 0.0141 | active-day beta 0.7809 (n=16 active days)
No-overlap ablation:   unconditional beta 0.0016 | active-day beta 0.0089 (n=133 active days)


### Coin-Concentration Diagnostics (2024-01-01 block)

`coin_concentration_diagnostics` (`stat_arb.portfolio`) is read-only instrumentation of the
already-built coin-netted portfolio -- it introduces no new coin-netting math, only summarizes
`coin_notional`/`equity` that `build_coin_netted_portfolio` already computed. Concentration is
only measured for the 2024-01-01 block since it is the only block with any selected pairs at
all under either model.

In [16]:
from stat_arb.portfolio import evaluate_coin_netted_portfolio, coin_concentration_diagnostics

richest_block = WF1_REBALANCE_DATES[0]
richest_block_end = WF1_REBALANCE_DATES[1] - pd.Timedelta(hours=4)
no_overlap_richest_pairs = [s['pair'] for s in NO_OVERLAP_SELECTIONS[richest_block]]
no_overlap_richest_pair_results = {s['pair']: NO_OVERLAP_ALL_RESULTS[(richest_block, s['pair'])]
                                     for s in NO_OVERLAP_SELECTIONS[richest_block]}
no_overlap_richest_portfolio, _ = evaluate_coin_netted_portfolio(
    no_overlap_richest_pair_results, no_overlap_richest_pairs, WF_PX.loc[:richest_block_end],
    CONFIG['COST_PER_LEG_BPS'], richest_block, richest_block_end, CONFIG['BARS_PER_YEAR'])
no_overlap_concentration = coin_concentration_diagnostics(no_overlap_richest_portfolio)

print(f"Max share of total portfolio gross exposure held by a single coin: "
      f"{no_overlap_concentration['max_share_of_total_gross_single_coin']:.1%}")
no_overlap_concentration['top5_coins_by_max_gross'].rename('max_gross_exposure_pct_of_equity').to_frame()

Max share of total portfolio gross exposure held by a single coin: 70.7%


,max_gross_exposure_pct_of_equity
LDOUSDT,0.468826
VETUSDT,0.065421
NEARUSDT,0.063089
DOGEUSDT,0.062581
UNIUSDT,0.061934


In [17]:
largest_gross_coin = no_overlap_concentration['largest_gross_coin_by_bar']
n_active_bars = largest_gross_coin.notna().sum()
ldo_share_of_active_bars = (largest_gross_coin == 'LDOUSDT').sum() / n_active_bars if n_active_bars else float('nan')
print(f"LDOUSDT is the single largest gross coin exposure on {ldo_share_of_active_bars:.1%} of "
      f"{n_active_bars} active bars in the 2024-01-01 block.")

LDOUSDT is the single largest gross coin exposure on 87.8% of 769 active bars in the 2024-01-01 block.


### LDOUSDT Pair-Cluster Diagnostics

All 10 no-overlap-selected pairs at 2024-01-01 share LDOUSDT as one leg (the funnel diagnostic
already established this). `coin_cluster_pair_table` joins the real diagnostic trail against the
real trades (same tuple-order normalization already used in `stat_arb.funnel` -- see that
module's docstring for why this join must never assume the two key conventions coincide).
`coin_cluster_return_correlation` answers whether these 10 pairs behave like diversified
relative-value opportunities or one concentrated LDOUSDT-centered factor trade.

In [18]:
from stat_arb.ablation import coin_cluster_pair_table, coin_cluster_return_correlation
from stat_arb.pair_selection import select_pairs_at_rebalance

_, _, no_overlap_pair_trail, _ = select_pairs_at_rebalance(
    richest_block, WF_PX, WF_PX.columns.tolist(), WF_CONFIG, max_pairs=WF_CONFIG['MAX_PAIRS_PER_REBALANCE'],
    collect_diagnostics=True, enforce_one_pair_per_coin=False)

ldo_cluster_table = coin_cluster_pair_table(richest_block, 'LDOUSDT', no_overlap_pair_trail,
                                              NO_OVERLAP_SELECTIONS, NO_OVERLAP_ALL_TRADES)
print(f"{len(ldo_cluster_table)} testable pairs involve LDOUSDT at {richest_block.date()}; "
      f"{int(ldo_cluster_table['fully_qualifying'].sum())} fully qualify, "
      f"{int(ldo_cluster_table['selected'].sum())} selected "
      f"({int((ldo_cluster_table['elimination_stage'] == 'max_pairs_cap').sum())} of the remainder "
      f"excluded only by the max-pairs cap, not by any statistical filter).")
ldo_cluster_table

32 testable pairs involve LDOUSDT at 2024-01-01; 15 fully qualify, 10 selected (5 of the remainder excluded only by the max-pairs cap, not by any statistical filter).


,pair,pvalue_adj,fully_qualifying,selected,elimination_stage,generated_entry_signal,n_executed_trades
0,LDOUSDT/OMUSDT,0.000358,True,True,None,True,2
1,DOTUSDT/LDOUSDT,0.000885,True,True,None,True,5
2,LDOUSDT/UNIUSDT,0.000885,True,True,None,True,2
3,LDOUSDT/NEARUSDT,0.000885,True,True,None,True,3
4,ATOMUSDT/LDOUSDT,0.000885,True,True,None,True,4
5,FTMUSDT/LDOUSDT,0.000885,True,True,None,True,2
6,DOGEUSDT/LDOUSDT,0.002648,True,True,None,True,2
7,LDOUSDT/VETUSDT,0.002648,True,True,None,True,4
8,FILUSDT/LDOUSDT,0.004610,True,True,None,True,5
9,LDOUSDT/SHIBUSDT,0.006083,True,True,None,True,2


In [19]:
ldo_cluster_corr = coin_cluster_return_correlation(richest_block, 'LDOUSDT', NO_OVERLAP_SELECTIONS, NO_OVERLAP_ALL_RESULTS)
offdiag = ldo_cluster_corr.values[np.triu_indices_from(ldo_cluster_corr.values, k=1)]
print(f'LDOUSDT-cluster pairwise 4h net-return correlation (full 2024-01-01 block, flat bars = 0 return): '
      f'mean={offdiag.mean():.3f}, median={np.median(offdiag):.3f}, min={offdiag.min():.3f}, max={offdiag.max():.3f}')
ldo_cluster_corr.round(2)

LDOUSDT-cluster pairwise 4h net-return correlation (full 2024-01-01 block, flat bars = 0 return): mean=0.163, median=0.135, min=-0.170, max=0.568


,OMUSDT/LDOUSDT,NEARUSDT/LDOUSDT,DOTUSDT/LDOUSDT,UNIUSDT/LDOUSDT,FTMUSDT/LDOUSDT,ATOMUSDT/LDOUSDT,VETUSDT/LDOUSDT,DOGEUSDT/LDOUSDT,FILUSDT/LDOUSDT,SHIBUSDT/LDOUSDT
OMUSDT/LDOUSDT,1.00,-0.00,-0.06,-0.09,-0.00,0.00,-0.03,-0.06,-0.07,-0.07
NEARUSDT/LDOUSDT,-0.00,1.00,0.28,0.13,0.03,0.00,0.28,0.11,0.05,0.05
DOTUSDT/LDOUSDT,-0.06,0.28,1.00,0.37,0.26,-0.17,0.36,0.30,0.35,0.33
UNIUSDT/LDOUSDT,-0.09,0.13,0.37,1.00,0.34,0.00,0.19,0.50,0.48,0.43
FTMUSDT/LDOUSDT,-0.00,0.03,0.26,0.34,1.00,0.00,0.10,0.30,0.28,0.44
ATOMUSDT/LDOUSDT,0.00,0.00,-0.17,0.00,0.00,1.00,-0.02,0.00,0.19,0.00
VETUSDT/LDOUSDT,-0.03,0.28,0.36,0.19,0.10,-0.02,1.00,0.15,0.12,0.19
DOGEUSDT/LDOUSDT,-0.06,0.11,0.30,0.50,0.30,0.00,0.15,1.00,0.35,0.57
FILUSDT/LDOUSDT,-0.07,0.05,0.35,0.48,0.28,0.19,0.12,0.35,1.00,0.37
SHIBUSDT/LDOUSDT,-0.07,0.05,0.33,0.43,0.44,0.00,0.19,0.57,0.37,1.00


In [20]:
from stat_arb.reporting import pair_level_contribution

no_overlap_pnl = pair_level_contribution(NO_OVERLAP_ALL_TRADES, NO_OVERLAP_ALL_RESULTS, CONFIG['COST_PER_LEG_BPS'])
print('Per-pair-instance P&L contribution -- every traded pair in the full OOS window under this '
      'ablation involves LDOUSDT:')
no_overlap_pnl.round(4)

Per-pair-instance P&L contribution -- every traded pair in the full OOS window under this ablation involves LDOUSDT:


,pair_instance,n_trades,sum_trade_return,win_rate,avg_holding_bars,pct_of_abs_pnl
0,VETUSDT/LDOUSDT @ 2024-01-01,4,0.2043,0.7500,48.7500,0.1095
1,ATOMUSDT/LDOUSDT @ 2024-01-01,4,0.0983,0.7500,64.2500,0.0527
2,DOTUSDT/LDOUSDT @ 2024-01-01,5,0.0824,0.6000,57.0000,0.0442
3,OMUSDT/LDOUSDT @ 2024-01-01,2,-0.0258,0.5000,41.0000,0.0138
4,NEARUSDT/LDOUSDT @ 2024-01-01,3,-0.0884,0.6667,37.6667,0.0474
5,DOGEUSDT/LDOUSDT @ 2024-01-01,2,-0.1730,0.0000,68.0000,0.0927
6,FILUSDT/LDOUSDT @ 2024-01-01,5,-0.1816,0.2000,39.8000,0.0973
7,UNIUSDT/LDOUSDT @ 2024-01-01,2,-0.2339,0.0000,66.5000,0.1253
8,FTMUSDT/LDOUSDT @ 2024-01-01,2,-0.2671,0.0000,22.5000,0.1432
9,SHIBUSDT/LDOUSDT @ 2024-01-01,2,-0.5110,0.0000,28.0000,0.2739


### Interpretation (B2.5)

Answering the pre-specified research questions directly from the measured results above, not
from intuition:

1. **10 pairs selected** at 2024-01-01 (15 fully qualify; the existing `MAX_PAIRS_PER_REBALANCE=10`
   cap -- left unchanged, per the ablation's scope -- excludes the 5 weakest by `pvalue_adj`:
   ICPUSDT, AVAXUSDT, BNBUSDT, ADAUSDT, APTUSDT, all paired with LDOUSDT).
2. **All 10 generate real entry signals** (100%).
3. **31 total executed trades** over the full OOS window (vs. 2 for baseline WF1) -- all in the
   single 2024-01-01 block; every other rebalance date has zero FDR-significant pairs under
   either selection method, so the ablation cannot manufacture trades where none survive the
   statistical screen.
4. **Time in market rises materially**: 1.5% (baseline) -> 14.1% (no-overlap), roughly 9x.
5. **Net performance deteriorates**: daily-252 Sharpe -0.068 -> -0.475; cumulative net return
   -3.8% -> -10.1%. (Max drawdown is smaller in magnitude, -18.6% -> -12.9% -- a diversification
   effect on drawdown that does not carry through to Sharpe or cumulative return.)
6. **Transaction costs rise**: 0.83% -> 1.26% of equity (turnover 4.15x -> 6.30x), consistent
   with 31 trades instead of 2.
7. **LDOUSDT concentration is severe**: gross exposure in LDOUSDT alone peaks at ~46.9% of
   portfolio equity (70.7% of the portfolio's own total gross exposure at that moment), and
   LDOUSDT is the single largest coin exposure on ~87.8% of all active bars in the block.
8. **Mixed-to-moderate positive correlation, not independent, not a single repeated trade**:
   the 10 pairs' 4h net-return series have a mean pairwise correlation of ~0.16 (median ~0.13;
   25 of 45 pairs above +0.10; only 1 of 45 meaningfully negative) -- well below what ten copies
   of the same trade would show, but clearly above pure independence. Combined with the
   concentration finding above, this reads as a **partially common LDOUSDT-centered exposure**
   expressed through different counterparties, not a set of ten diversified relative-value bets.
9. **BTC beta**: unconditional beta drops further toward zero (0.0141 -> 0.0016); active-day
   beta also drops sharply (0.78, n=16 active days -> 0.009, n=133 active days). The larger
   active-day sample under the ablation is itself a byproduct of trading far more, not
   independent evidence of market-neutrality -- and WF1's own n=16 active-day estimate was
   already flagged as too small to trust on its own.
10. **Based on the measured results, not intuition**: removing the one-pair-per-coin constraint
    added 29 trades and ~12.6 points of time-in-market but *worsened* both Sharpe and cumulative
    net return, *raised* transaction costs, and produced a portfolio dominated by a single
    coin's exposure for most of its active life. On this one richest-block episode, the
    constraint appears to have been suppressing **concentration risk**, not a materially better
    opportunity set. This is a single rebalance block and 31 trades -- an even smaller sample
    than WF1/WF2's own low-trade-count caveat already covers -- so this is reported as a
    single-episode exploratory finding, not a robust conclusion, and `WF1_NO_OVERLAP_CONSTRAINT`
    is not adopted as a new primary specification.

## WF2 -- Rolling Entry Hedge Sizing

In [21]:
from stat_arb.walkforward import run_wf2_full

WF2_FULL_FRAME, WF2_ALL_RESULTS, WF2_ALL_TRADES = run_wf2_full(
    WF1_SELECTIONS, WF1_REBALANCE_DATES, CONFIG['OOS_END'], WF_PX, CONFIG)

wf2_row = standardized_wf1_summary('WF2_ROLLING_ENTRY_HEDGE_RATIO', WF2_FULL_FRAME, WF2_ALL_RESULTS,
                                    WF2_ALL_TRADES, WF_PX, CONFIG['OOS_START'], CONFIG['OOS_END'])

wf1_vs_wf2_df = pd.DataFrame([WF1_ROW, wf2_row]).set_index('model')
print('WF1 vs. WF2 -- OOS standardized comparison (LOW TRADE COUNT: interpret with caution):')
wf1_vs_wf2_df.round(4).T

[NOTE] Peak realized gross_exposure = 1.146 at 2024-02-03 00:00:00 (intra-trade mark-to-market drift on fixed quantities, not a sleeve-allocation violation -- sleeve weights themselves sum to <=1.0 by construction).
WF1 vs. WF2 -- OOS standardized comparison (LOW TRADE COUNT: interpret with caution):


model,WF1_SEMIANNUAL_REESTIMATION,WF2_ROLLING_ENTRY_HEDGE_RATIO
daily_sharpe_252,-0.0679,-0.0185
ann_volatility,0.0931,0.0854
cum_net_return,-0.0380,-0.0188
cum_gross_return,-0.0299,-0.0107
max_drawdown,-0.1858,-0.1563
n_trades,2.0000,2.0000
n_pairs_traded,1.0000,1.0000
win_rate,0.5000,0.5000
avg_trade_return,-0.0129,-0.0040
median_trade_return,-0.0129,-0.0040


## Interpretation and limitations

WF1 trades only a handful of times in the full OOS window; WF2 differs from WF1 only in
entry-time hedge sizing and trades the same instances. With this few trades, any Sharpe/return
comparison between WF1, WF2, S0, and S1 is not a robust statistical conclusion -- it is reported
with this caveat stated explicitly, per `CLAUDE.md`'s low-trade-count requirement.

In [22]:
wf1_wf2_tables_dir = PROJECT_ROOT / 'results' / 'tables'
wf1_wf2_tables_dir.mkdir(parents=True, exist_ok=True)
with open(wf1_wf2_tables_dir / 'wf1_wf2_rows.json', 'w') as f:
    json.dump({'WF1_SEMIANNUAL_REESTIMATION': WF1_ROW, 'WF2_ROLLING_ENTRY_HEDGE_RATIO': wf2_row}, f, indent=2, default=str)
print(f'Saved WF1/WF2 rows to {wf1_wf2_tables_dir / "wf1_wf2_rows.json"}')

Saved WF1/WF2 rows to C:\Users\kuhfu\OneDrive\A_Nils\Studium\Quant\WSQ\My Project\Stat Arb - Pair Trading\results\tables\wf1_wf2_rows.json
